In [ ]:
"""Today is one of the most important days in month 2. Today, my system stops blindly
trusting it's own retrieval. Am going to build two things: a Retriever LLM that checks
the agent's answer against the retrieved chunks and flags hallucinations, and 
Corrective RAG logic that automatiaclly rejects low-confidence retrieval results
and falls back to web search. Together these make my system self-aware about when 
it is likely to be wrong.

The Retriever LLM: A separate LLM call that receives three things. The original question,
the retrieved chunks, and the generated answer and returns a structured verdict: PASS,
FAIL or UNCERTAIN, plus a one-sentence reason. If it returns FAIL, the answer is sent
back for a rewrite with the reviewer's critique attached.

The Rewrite Loop: Cap rewrites at 2 attempts to avoid infinite loops. If the reviewer still returns
FAIL, afetr two rewrites, the system returns the best attempt with a warning flag
rather than silently returning a bad answer.

Corrective RAG: Before the answer is even generated, score the retrieved chunks for 
relevance. If the top chunk's relevance score is below a threshold(0.5),
automatically discard all retrieved chunks and fall back to Tavily web search instead.

Combining Both: The full pipeline is Retrieve -> score relevance -> if low score 
use web fallback -> generate answer -> reviewer checks -> rewrite if needed


"""


# importing necessary libraries

import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from langchain_community.tools.tavily_search import TavilySearchResults

load_dotenv(find_dotenv())

client = Groq()
Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)
webSearch = TavilySearchResults(max_results = 3)
    

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rodne\AppData\Local\Temp\ipykernel_2468\1150903352.py:53: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  webSearch = TavilySearchResults(max_results = 3)


In [2]:
# LOADING EXISTING VECTOR STORE

# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')

🤖🛩️ Vector Store connection established ⚡


In [ ]:
# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # results is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

def format_chunks(nodes_with_scores: list) -> str:
    return "\n\n---\n\n".join([r.node.get_content() for r in nodes_with_scores])
        

In [17]:
# CRAG -> Retrieval quality gate

def corrective_retrieve(query: str) -> tuple[str, str]:
    """
    Returns (context_text, source) where source is 'local' or 'web'.
    Applies CRAG logic: low confidence -> discard local, use web fallback.
    """

    docs, top_score = retrieve_with_confidence(query)

    if top_score < RELEVANCE_THRESHOLD or not docs:
        print("🤥 CRAG: Low retrieval confidence - falling back to web search")
        webResults = webSearch.invoke(query)
        context = "\n\n".join([r["content"] for r in webResults])
        return context, "web"
    
    else:
        print("👍 CRAG: Retrieval confidence acceptable - using local docs")
        return format_chunks(docs), "local"

In [ ]:
# ANSWER GENERATOR

GENERATOR_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the user's question using ONLY the context provided.
If the context does not contain enough information to answer, say exactly:
'I cannot find sufficient information in the provided context.' 
Be specific - include numbners, percentages, and fiscal year references where available.

"""

def generate_answer(question: str, context: str, critique: str = "") -> str:
    critiqueBlock = ""
    if critique:
        critiqueBlock = f"\n\nPrevious answer was rejected for this reason: {critique}\nPlease rewrite addressing this critique."

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = [
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
                f"{critiqueBlock}"
            )}
        ],

        temperature= 0,
    )

    return response.choices[0].message.content

In [ ]:
# THE REVIEWER LLM

REVIEWER_SYSTEM_PROMPT = """You are a strict factual reviewer for a financial RAG system.
You will receive a question, the source context, and a generated answer.

Your job is to check:
1. Does the answer contain any claims NOT supported by the context? (hallucination)
2. Does the answer actually address the question asked?
3. Are numbers, percentages, and figures accurate relative to the context?

Respond in EXACTLY this format:
Verdict: <PASS or FAIL>
Reason: <one sentence explaining your verdict>

PASS means the answer is faithful to the context and addresses the question,
FAIL means the answer contains unsupported claims, wrong figuress, or avoids the question.


"""

def review_answer(question: str, context: str, answer: str) -> tuple[str, str]:
    """Returns (verdict, reason) where verdict is a PASS or FAIL."""
    response = client.chat.completions.create(
        model = 'openai/gpt-oss-120b',
        messages = [
            {"role": "system", "content": REVIEWER_SYSTEM_PROMPT},
            {"role": "user", "content":(
                f"Question: {question}\n\n"
                f"Source Context:\n{context}\n\n"
                f"Generated Answer:\n{answer}"

            )}
        ],
        temperature= 0
    )

    raw = response.choices[0].message.content
    verdict_match = re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match =re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f"😮‍💨 Reviewer verdict: {verdict} - {reason}")
    return verdict, reason


In [ ]:
# THE FULL SELF CORRECTING CRAG PIPELINE

def run_crag_pipeline(question: str, max_rewrites: int = 2) ->dict:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'='*60}")

    startedAt = datetime.now().isoformat()

    # STEP 1: CRAG retrieval with confidence gate
    context, source = corrective_retrieve(question)

    # STEP 2: Generate Initial Answer
    print("\n 🦾 Generating initial answer...")
    answer = generate_answer(question, context= context)
    print(f"Answer: {answer}\n")

    # SECONDARY GATE to catch false-positive vector scores
    if answer and "I cannot find sufficient information" in answer and source == "local":
        print("🤥 CRAG: Local docs failed to answer despite high vector score. Forcing web fallback...")
        webResults = webSearch.invoke(question)
        context = "\n\n".join([r["content"] for r in webResults])
        source = "web"

        print("👍Generating answer from web context...")
        answer = generate_answer(question, context= context)
        print(f"Web Fallback Answer: {answer}\n")

    # STEP 3: Reviewer Loop
    attempts = 0
    verdict = 'FAIL'
    critique = ""
    history = []

    while verdict == 'FAIL' and attempts < max_rewrites:
        verdict, critique = review_answer(question= question, context= context, answer= answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

        if verdict == 'FAIL':
            attempts += 1
            if attempts < max_rewrites:
                print(f"\n🔁 Rewriting (attempt {attempts})...")
                answer = generate_answer(question, context, critique)
                print(f"Rewritten Answer: {answer}\n")
            else:
                print("🤥 Max rewrites reached - returning best attempt with warning")

    # final verrdict check if we exited the loop with PASS 
    if verdict != 'FAIL':
        verdict, critique = review_answer(question, context, answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

    result = {
        "question": question,
        "retrieval_source": source,
        "final_answer": answer,
        "fianl_verdict": verdict,
        "rewrite_attempts": attempts,
        "review_history": history,
        "started_at":startedAt,
        "finished_at": datetime.now().isoformat()
    }


    print(f"\n{'='*60}")
    print(f"👍 Final Answer ({verdict} after {attempts} rewrite(s)):")
    print(answer)
    print(f"{'='*60}\n")

    filename = f"traces/day11_crag_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent= 2)
    print(f"📀 Saved to {filename}")

    return result


In [21]:
# EVALUATION

# Question 1: High confidence retrieval - retriever should PASS first attempt

run_crag_pipeline(
    "What was apple's total net sales for fiscal year 2024"
    "and how does it compare to fiscal year 2023"
)

# Question 2: Borderline retrieval - tests whether reviewer catches vague answers

run_crag_pipeline(
    "What were the primary risk factors Apple disclosed in the FY2024 10-K"
    "related to global macroeconomic conditions?"
)

# Question 3: Outside the 10-K, CRAG must fallback to web Search 
run_crag_pipeline(
    "What is Apple's current stock price and market capitalisation today?"
)


Question: What was apple's total net sales for fiscal year 2024and how does it compare to fiscal year 2023

📣 Top retrieval score : 0.875 (threshold: 0.5)
👍 CRAG: Retrieval confidence acceptable - using local docs

 🦾 Generating initial answer...
Answer: Apple’s total net sales were **$391,035 million for fiscal year 2024**.  
For fiscal year 2023, total net sales were **$383,285 million**.

**Comparison:** FY 2024 net sales were **$7,750 million higher** than FY 2023, representing an increase of roughly **2 %** year‑over‑year.

😮‍💨 Reviewer verdict: PASS - The answer correctly reports the total net sales for FY 2024 and FY 2023 from the context and accurately calculates the dollar and percentage difference.
😮‍💨 Reviewer verdict: PASS - The answer correctly reports FY 2024 and FY 2023 total net sales figures from the context and accurately calculates the $7,750 million (≈2 %) increase.

👍 Final Answer (PASS after 0 rewrite(s)):
Apple’s total net sales were **$391,035 million for fisca

{'question': "What is Apple's current stock price and market capitalisation today?",
 'retrieval_source': 'web',
 'final_answer': 'Apple’s stock is currently trading at **$282.50 per share**.  \nIts market capitalisation as of\u202fJune\u202f28\u202f2026 is **approximately $4.168\u202ftrillion** (about $4.13\u202ftrillion in the earlier snapshot).',
 'fianl_verdict': 'PASS',
 'rewrite_attempts': 0,
 'review_history': [{'attempt': 1,
   'answer': 'Apple’s stock is currently trading at **$282.50 per share**.  \nIts market capitalisation as of\u202fJune\u202f28\u202f2026 is **approximately $4.168\u202ftrillion** (about $4.13\u202ftrillion in the earlier snapshot).',
   'verdict': 'PASS',
   'critique': 'The answer correctly reports the stock price and market capitalisation exactly as given in the context and directly answers the question.'},
  {'attempt': 1,
   'answer': 'Apple’s stock is currently trading at **$282.50 per share**.  \nIts market capitalisation as of\u202fJune\u202f28\u202